# Proyecto Oráculo · Notebook de evaluación (ADAG1, Grupo 1)

Evalúa la configuración de `entrega.json` (en la raíz del repo) sobre los **datos de prueba reservados**. Se corre de arriba abajo **sin editar nada**:

1. Celda 1: instala y clona el repo. Al terminar hay que **reiniciar la sesión**.
2. Celda 2: carga `entrega.json` y los datos de prueba. Si no existe `/content/datos_prueba.json`, pide subir el archivo (JSON con el mismo formato que `datos_visibles.json`).
3. Celda 3: carga el modelo, evalúa y muestra la precisión.

Necesita **GPU T4**: *Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU*.

## 1 · Instalar y clonar el repo · *al terminar, reinicia el entorno de ejecución*

In [ ]:
# bitsandbytes: checkpoints 4-bit. nltk/spacy/emoji/langdetect: los usa el
# verificador de open-instruct. Copia de la celda 1 del notebook del curso, pero
# clonando el fork del grupo en vez de bajar los .py sueltos.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q

%cd /content
# oraculo.py busca los verificadores en /content/open-instruct.
!test -d open-instruct || git clone -q https://github.com/allenai/open-instruct
# El repo del grupo trae oraculo.py, ayudas.py, datos_visibles.json, busqueda/ y entrega.json.
!test -d ADA-2026-2-GP01 && git -C ADA-2026-2-GP01 pull -q || git clone -q https://github.com/Lenuard/ADA-2026-2-GP01

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")

## 2 · Configuración y datos de prueba

In [ ]:
%cd /content/ADA-2026-2-GP01
import json
import os
import sys

sys.path.insert(0, "/content/ADA-2026-2-GP01")
from busqueda import validar_entrega

if not os.path.exists("entrega.json"):
    raise FileNotFoundError("el repo no trae entrega.json todavía: falta correr notebooks/busqueda.ipynb")
with open("entrega.json") as f:
    ENTREGA = json.load(f)
validar_entrega(ENTREGA)
CONFIG = ENTREGA["config"]
print("grupo:", ENTREGA["grupo"], "· semana:", ENTREGA["semana"])
print("config:", CONFIG)

RUTA_DATOS = "/content/datos_prueba.json"
if not os.path.exists(RUTA_DATOS):
    from google.colab import files
    print(f"No está {RUTA_DATOS}: suba el JSON de datos de prueba.")
    subidos = files.upload()
    nombre = next(iter(subidos))
    with open(RUTA_DATOS, "wb") as f:
        f.write(subidos[nombre])
with open(RUTA_DATOS) as f:
    DATOS = json.load(f)
print(f"{len(DATOS)} instancias de prueba")

## 3 · Evaluar

In [ ]:
from collections import Counter

from ayudas import cargar_modelo
from oraculo import Oraculo

# El mismo modelo con el que se buscó la config.
MODELO = "qwen17b"
modelo = cargar_modelo(MODELO)
oraculo = Oraculo(modelo, DATOS, cache="cache_evaluacion.json", lote=16, presupuesto=32_000_000)

r = oraculo.evaluar(CONFIG, DATOS, semilla=1)
print(f"\nPRECISIÓN EN PRUEBA: {r.precision:.1%}  ({r.n - len(r.trazas)}/{r.n} instancias)")
print("familias que más fallaron:")
for fam, veces in Counter(t["violo"] for t in r.trazas).most_common(10):
    print(f"  {veces:3}  {fam}")